# Explanation import process  

In [1]:
import yfinance as yf
import pandas as pd


# initial population of the db 
names_sp = pd.read_csv("sp500_companies.csv")["Symbol"]


In [2]:
start_time = "2025-01-01"
end_time = "2026-01-01"

df= yf.download('msft aapl goog', start = start_time, end = end_time, actions=True, group_by='ticker')


[*********************100%***********************]  3 of 3 completed


## DataControlling for our example

### MultiIndex

In [3]:
if isinstance(df.columns, pd.MultiIndex):
    print("Multiindex")

Multiindex


### Sort index and drop duplicates

In [4]:
df = df.sort_index()
if isinstance(df.index, pd.DatetimeIndex):
    print("Datetime")

df = df[~df.index.duplicated(keep="first")]

Datetime


In [5]:
df_misses = df[df.isna().any(axis=1)]

## OHLC High > max(Open, Close), Low< min(Open,Close),

In [6]:
high = df.xs('High', axis=1, level=1)
high = high.sort_index(axis=1)
low = df.xs('Low', axis=1, level=1)
open_ = df.xs('Open', axis=1, level=1)
close = df.xs('Close', axis=1, level=1)
testdf = pd.concat([open_, close], axis=1, keys=['Open','Close']).T.groupby(level=1).max().T
invalid_high = high.where(high < testdf)


In [7]:
AAPLE = yf.Ticker("aapl")
a_info = AAPLE.info

#shows dividents splits 
a_actions = AAPLE.actions
from datetime import datetime,timedelta

#a_min = AAPLE.history(start=datetime.now()-timedelta(days=60), end=datetime.now(), interval="5m")

#Price to earnings 
a_pe = a_info["forwardPE"]
#dividentrate 
a_div = a_info["dividendRate"]
#check for 1 min data
start_time = datetime.now()-timedelta(days=3)
end_time = datetime.now()
#AAPLE.history(start = start_time, end=end_time, interval="1min")
a_short_hist = AAPLE.history(period="5d", interval="1m")

In [8]:
# Inspect important fields in a_info for prioritization
selected_fields = [
    "symbol", "shortName", "longName", "quoteType", "currency", "exchange",
    "sector", "industry", "country", "fullTimeEmployees",
    "marketCap", "enterpriseValue", "totalRevenue", "ebitda", "grossMargins", "operatingMargins", "profitMargins",
    "trailingPE", "forwardPE", "pegRatio", "priceToBook", "trailingEps", "forwardEps",
    "returnOnEquity", "returnOnAssets", "debtToEquity", "currentRatio", "quickRatio",
    "dividendRate", "dividendYield", "payoutRatio", "exDividendDate",
    "fiftyTwoWeekHigh", "fiftyTwoWeekLow", "fiftyDayAverage", "twoHundredDayAverage",
    "beta", "averageVolume", "sharesOutstanding", "freeCashflow"
]

print(f"Total keys in a_info: {len(a_info)}")
print("\nSelected field snapshot (present keys):")
for k in selected_fields:
    if k in a_info:
        print(f"- {k}: {a_info.get(k)}")

missing = [k for k in selected_fields if k not in a_info]
print(f"\nMissing from selected list: {len(missing)}")
if missing:
    print(missing)

print("\nFirst 80 available keys:")
for k in sorted(list(a_info.keys()))[:80]:
    print(k)


Total keys in a_info: 183

Selected field snapshot (present keys):
- symbol: AAPL
- shortName: Apple Inc.
- longName: Apple Inc.
- quoteType: EQUITY
- currency: USD
- exchange: NMS
- sector: Technology
- industry: Consumer Electronics
- country: United States
- fullTimeEmployees: 150000
- marketCap: 3676245065728
- enterpriseValue: 3695648702464
- totalRevenue: 435617005568
- ebitda: 152901992448
- grossMargins: 0.47325
- operatingMargins: 0.35374
- profitMargins: 0.27037
- trailingPE: 31.660759
- forwardPE: 26.906572
- priceToBook: 41.700565
- trailingEps: 7.9
- forwardEps: 9.29587
- returnOnEquity: 1.5202099
- returnOnAssets: 0.24377
- debtToEquity: 102.63
- currentRatio: 0.974
- quickRatio: 0.845
- dividendRate: 1.04
- dividendYield: 0.42
- payoutRatio: 0.1304
- exDividendDate: 1770595200
- fiftyTwoWeekHigh: 288.62
- fiftyTwoWeekLow: 169.21
- fiftyDayAverage: 262.7076
- twoHundredDayAverage: 245.5843
- beta: 1.116
- averageVolume: 48098727
- sharesOutstanding: 14681140000
- freeCash